## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

### **Этап 1: Подготовка данных**

**Задача:** На этом этапе вам надо собрать структурированный список словарей с информацией о лауреатах и призах. Приведем начало нужного нам датасета.

``` {code:python}
[{'id': '745',
  'name': 'A. Michael Spence',
  'gender': 'male',
  'birth_year': 1943,
  'country_birth': 'USA',
  'country_now': 'USA',
  'prizes_relevant': [{'prize_amount': 10000000,
    'prize_amount_adjusted': 15547541,
    'award_year': 2001,
    'category_en': 'Economic Sciences',
    'prize_status': 'received'}],
  'type_': 'person'},
 {'id': '102',
  'name': 'Aage N. Bohr',
  'gender': 'male',
  'birth_year': 1922,
  'country_birth': 'Denmark',
  'country_now': 'Denmark',
  'prizes_relevant': [{'prize_amount': 630000,
    'prize_amount_adjusted': 4304697,
    'award_year': 1975,
    'category_en': 'Physics',
    'prize_status': 'received'}],
  'type_': 'person'}]
  ```

Поскольку информация о лауреатах-людях и лауреатах-организациях структурирована по-разному, а также из-за того, что часть полей в json могут отсутстввовать, необходимо реализовать функции для процессинга словарей, которые будут достаточно обобщаемыми для обоих типов данных.

**Что нужно сделать:**

На входе информация о конкретном лауреате представляет из себя кучу вложенных друг в друга словарей. Из этой структуры нам надо получить плоский словарь вида {atribute: value}, где atribute --- это один из небольшого множества интересных нам атрибутов. При этом в исходном файле 
1. пути к атрибутам могут иметь разную длину
2. атрибуты могут отсутствовать
3. атрибуты могут нуждаться в дополнительной обработке (например, поле id в исходных словарях записано как строка.)

Поскольку наивная реализация такого обработчика оказывается довольно громоздкой, вам нужно создать файл json_dict_processing.py, в котором реализован высокоуровневый функционал для решения задачи "обработать вложенный json-словарь"

Главное требование к интерфейсу: 
по (вложенному) словарю о лауреате и конфигу (см ниже) надо получить плоский словарь вида {atribute: value}.

Достичь этого можно, например, таким набором функций

* `extract_nested_value(obj, keys)` - извлекает значение из вложенных структур данных по цепочке ключей

* `process_dictionary_with_config(dictionary, config)` - обрабатывает один словарь согласно словару-конфигу (формат конфига см ниже.)

* `process_list_of_dicts_with_config(list_of_dicts, config)` - обрабатывает список словарей, эта функция нужна чтобы обработать список призов.

* `create_processor(config, list_processor=False)` - создает готовый процессор с предзагруженной конфигурацией. В случае, если list_processor == True, внутрь процессора подается process_list_of_dicts_with_config, иначе process_dictionary_with_config.

**Замечание насчет последней функции.**

Планируемое применение этой функции примерно такое:
`laureate_processor = create_processor(PERSON_CONFIG, list_processor=False)`

То есть, create_processor возвращает функцию, которая реализует функционал process_dictionary_with_config но с уже заполненным конфигом. Это повышает читаемость кода и снижает риск перепутать конфиги из-за дублирования кода.


Помимо этого нужно создать файлы-конфиги для данных по призам, лауреатам-людям, и лауреатам-организациям.

КОНФИГ для функций process_dictionary_with_config и process_list_of_dicts_with_config имеет следующий вид
```python
  {'атрибут_который_берем_сырым': ['путь', 'к', 'данным'],  # или
  'атрибут_который_нужно_обработать': (['путь', 'к', 'данным'], функция_обработки)}
  ```
В случае, если атрибут можно взять в итоговый датасет сырым, в values() словаря-конфига будет список path_list.
Если же атрибут еще надо обработать, для него в поле config[attribute] будет tuple вида (path_list, function).

Примеры атрибутов, которые можно взять сырыми: имя, размер приза. Примеры атрибутов, которые нужно обработать: год вручения приза (в сырых данных есть только полная дата, записанная как строка)

После того, как будет создан core.py, создайте конфиги laureates_configs.py и prizes_configs.py

Постарайтесь провалидировать код с помощью assert-ов, вызываемых в ноутбуке или в .py-файлах.

пример использования: 

assert 1 == True

**Валидация**

В вашем случае assert должен взять простой словарик с одним вложенным полем и какой-нибудь конфиг-файл.
Также убедитесь, что при обработке данных собственно по лауреатам в них нет странностей (смотрите ниже) 

**Атрибуты для ПРИЗА (CONFIG_PRIZE, prizes_configs.py)**

- `prize_amount` - сумма приза → `prizeAmount`
- `prize_amount_adjusted` - скорректированная сумма приза → `prizeAmountAdjusted`
- `award_year` - год вручения приза (преобразуется в целое число) → `awardYear`
- `category_en` - категория на английском → `category.en`
- `prize_status` - статус приза → `prizeStatus`

**Атрибуты для ЧЕЛОВЕКА-ЛАУРЕАТА (CONFIG_PERSON, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - имя лауреата → `knownName.en`
- `gender` - пол → `gender`
- `birth_year` - год рождения (извлекается из даты) → `birth.date`
- `country_birth` - страна рождения → `birth.place.country.en`
- `country_now` - текущая страна → `birth.place.countryNow.en`
- `prizes_relevant` - список призов лауреата → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Атрибуты для ОРГАНИЗАЦИИ-ЛАУРЕАТА (CONFIG_ORG, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - название организации → `orgName.en`
- `founded_year` - год основания (извлекается из даты) → `founded.date`
- `country_founded` - страна основания → `founded.place.country.en`
- `country_now` - текущая страна → `founded.place.countryNow.en`
- `prizes_relevant` - список призов организации → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Вспомогательные функции:**

- `process_year(year_string)` - извлекает год из строки даты (формат "YYYY-MM-DD"). Функция находится в файле laureates_configs.py
- `prize_processor()` - создает процессор для обработки списков призов. Функция находится в файле prizes_configs.py
- `person_processor()` - создает процессор для данных о людях-лауреатах. Функция находится в файле laureates_configs.py
- `org_processor()` - создает процессор для данных об организациях-лауреатах. Функция находится в файле laureates_configs.py

In [1]:
import json

with open('laureates.json', encoding='utf-8') as file:
    laureates = json.load(file)

#У меня что-то юпитер отказывался читать файл по api c из-за блокеров со строны рабочего ноутбука, поэтому пришлось грузить локально

In [3]:
from laureates_configs import process_orgs, process_persons  # реализуйте эти функции согласно описанной выше логике

def get_laureate_data(laureate):
    """
    Определите тип лауреата and соотв образом обработайте данных.
    
    Args:
        laureate: Dictionary with laureate data
        
    Returns:
        Processed laureate data with added 'type_' field ('person' or 'org')
        
    Determines type by:
    - 'knownName' key → person (uses process_persons)
    - 'orgName' key → org (uses process_orgs)
    """
    # YOUR CODE HERE
    if 'knownName' in laureate:
        ans = process_persons(laureate)
        ans['type_'] = 'person'
    elif 'orgName' in laureate:
        ans = process_orgs(laureate)
        ans['type_'] = 'org'
    else:
         raise ValueError('ПРостите, сегодня без типа лауреата..')

    return ans


laureates_data = []

for laureate in laureates['laureates']:
    laureates_data.append(get_laureate_data(laureate))

In [11]:
example = {'id': '8', 'knownName': {'en': 'Nikita'}}
ans = get_laureate_data(example)

assert ans['id'] == 8
assert ans['name'] == 'Nikita'
assert ans['type_'] == 'person'
assert len(laureates_data) == len(laureates['laureates'])

print('Проверки пройдены успошено!')

Проверки пройдены успошено!




### **Этап 2: EDA лауреатов**

EDA или Exploratory Data Analysis - первый шаг при работе с любыми данными.

Чтобы начать работать с этой секцией, проведите проведите чисто технический анализ собранных данных.

1. Сколько всего записей?
2. Все ли значения полей заполнены?
3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?
5. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

При реализации ответов на эти вопросы, пользуйтесь теми же принципами, что описаны выше. Можете начать с плохого хакерского кода, однако в таком случае в маркдауне пропишите, как будете его обобщать и разносить по функциям и модулям. (об этом ниже)

В основном EDA пользуйтесь аккуратным кодом, разбитым на функции и файлы.

In [ ]:
# YOUR CODE HERE
from prizes_configs import CONFIG_PRIZE
from dataset_helpers import filter_rows, group_by_attributes
from aggregations import count_unique, describe_values, top_n
from atribute_manager import add_attribute
def show_missing(title, rows, values):
    counts = {}
    shares = {}
    for value in values:
        missing = 0
        for row in rows:
            if (value not in row) or (row[value] in (None,'',[])):
                missing += 1
        counts[value] = missing
        if rows:
            shares[value] = round(missing*100/len(rows), 2)
        else:
            shares[value] = None
    print(title)
    print('Количество пропусков', counts)
    print('Доля пропусков', shares)

people = []
organizations = []
prizes = []
for row in laureates_data:
    if row['type_'] == 'person':
        people.append(row)
    else:
        organizations.append(row)
    row_prizes = row['prizes_relevant']
    if row_prizes is not None:
        for prize in row_prizes:
            prizes.append(prize)

print('Всего лауреатов:', len(laureates_data))
common_values = ['id', 'name', 'country_now', 'prizes_relevant', 'type_']
people_values = ['gender', 'birth_year', 'country_birth']
org_values = ['founded_year', 'country_founded']
show_missing('Общие поля', laureates_data, common_values)
show_missing('Люди', people, people_values)
show_missing('Организации', organizations, org_values)
show_missing('Призы', prizes, CONFIG_PRIZE)

Всего лауреатов: 1028
Общие поля
Количество пропусков {'id': 0, 'name': 0, 'country_now': 7, 'prizes_relevant': 0, 'type_': 0}
Доля пропусков {'id': 0.0, 'name': 0.0, 'country_now': 0.68, 'prizes_relevant': 0.0, 'type_': 0.0}
Люди
Количество пропусков {'gender': 0, 'birth_year': 0, 'country_birth': 3}
Доля пропусков {'gender': 0.0, 'birth_year': 0.0, 'country_birth': 0.3}
Организации
Количество пропусков {'founded_year': 1, 'country_founded': 4}
Доля пропусков {'founded_year': 3.57, 'country_founded': 14.29}
Призы
Количество пропусков {'prize_amount': 0, 'prize_amount_adjusted': 0, 'award_year': 0, 'category_en': 0, 'prize_status': 0}
Доля пропусков {'prize_amount': 0.0, 'prize_amount_adjusted': 0.0, 'award_year': 0.0, 'category_en': 0.0, 'prize_status': 0.0}


In [14]:
ids = []
for row in laureates_data:
    ids.append(row['id'])
ids.sort()
duplicates = set()
gaps = []
for i in range(1,len(ids)):
    left = ids[i-1]
    right = ids[i]
    if left == right:
        duplicates.add(right)
    elif right-left > 1:
        gaps.append((left, right))

print('Повторяющиеся id', sorted(duplicates))
print('Разрывы', gaps)
print('Минимальный и максимальный id', ids[0], ids[-1])

for row in laureates_data:
    if row['id'] in (ids[0],ids[-1]):
        print(row['id'], row['name'])
        prizes = row['prizes_relevant']
        if prizes is not None:
            for prize in prizes:
                print(prize['award_year'])

Повторяющиеся id []
Разрывы [(6, 8), (93, 95), (169, 172), (255, 257), (468, 470), (503, 505), (520, 523), (541, 543), (569, 571), (590, 592), (594, 596), (597, 600), (610, 613), (615, 617), (631, 633), (635, 637), (645, 647), (655, 657), (783, 786), (787, 789), (858, 861), (896, 903), (949, 958), (1009, 1011)]
Минимальный и максимальный id 1 1070
1070 Michael Woodford
2026
1 Wilhelm Conrad RГ¶ntgen
1901



Поскольку это курс по Python, а не по статистике или визуализации данных, мы не задаем жесткие рамки того, что вы должны проанализировать.
Однако это отличная возможность представить себя в роли разработчика базы данных или, например, библиотеки pandas.

В коде нужно соблюдать следующие правила:

1. должны быть реализованы  несколько способов работы с данными (среднее/медианное/максимальное значение метрики, количество примеров, вывод топ-N объектов по значению метрики).
2. анализ выбранных вами тем должен включать в себя анализ метрик в тотале, по категориям и в динамике по годам или десятилетиям.
3. проанализировать стоит 2-3 темы. Вы можете использовать темы ниже или взять свои.
2. вывод ноутбука должен быть понятен обывателю. Это может быть визуализация в matplotlib/seaborn. Сопровождайте вывод markdown-ячейками с вашими  мыслями, пояснениями и инсайтами, которые удалось извлечь.
3. Нельзя использовать pandas и numpy, но можно вдохновляться их интерфейсом и функционалом при дизайне ваших функций.


Из пункта 2 следует, что вам придется работать не только с атрибутами лауреатов, но и самих призов.
Информацию о призах нужно доставать и записывать как атрибут уже лауреата. Функции, работающие с призами (например, факт того, что лауреат отказался от какого-то из призов) можно реализовать в ноутбуке, но если вы создаете более общие вспомогательные функции, выносите их в отдельные модули.

Мы рекомендуем следующую структуру кода:

* `dataset_helpers.py` в котором реализованы фильтрация списка словарей (плоских) по условиям (аналог sql-ного where), а также аналог sql-ного групбая. А также функция для применения агрегаций к списку словарей
* `aggregations.py` в котором реализованы функции расчета среднего/медианы/других эвристик и мб вывод топ n-значений по какой-то метрике.
* `atribute_manager.py` в котором реализован переиспользуемый интерфейс для добавления (и мб удаления) полей.

Замечание насчет group-by. 

На игрушесном примере она может работать примерно так:
```
    Входные данные: [
        {'name': 'Alice', 'country': 'USA', 'age': 25},
        {'name': 'Bob', 'country': 'UK', 'age': 30},
        {'name': 'Charlie', 'country': 'USA', 'age': 25}
    ]
    
    group_by_attributes(data, ['country', 'age']) → {
        ('USA', 25): [
            {'name': 'Alice', 'country': 'USA', 'age': 25},
            {'name': 'Charlie', 'country': 'USA', 'age': 25}
        ],
        ('UK', 30): [
            {'name': 'Bob', 'country': 'UK', 'age': 30}
        ]
    }
```

Замечание насчет agregations.py
1. Агрегации соответственно должны применяться к спискам вида [
    {'name': 'Alice', 'country': 'USA', 'age': 25},
    {'name': 'Charlie', 'country': 'USA', 'age': 25}
]
2. Пользуйтесь модулем statistics из стандартной библиотеки питона.


Эта структура одна из возможных для организации вашего кода, главное, чтобы эти достаточно общие функции блыи переиспользованы

----------

При анализе данных вам понадобятся более конкретные функции, например filter_by_num_prizes или output_metric_by_category. Их пишите прямо в ноутбуке и сопровождайте коментами.

Ниже мы приводим некоторые примеры вопросов, которые может быть интересно задать к данным, но если у вас есть желание, можете поисследовать датасет с любой интересной вам стороны!


**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов
- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

**2.2. Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

**2.3. Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?
- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

**2.4. Миграции лауреатов**
- У скольки лауреатов исходная страна и страна на момент вручения нобелевской премии различаются? Всегда ли это различие объясняется миграцией? Добавьте вспомогательное поле 'country_of_origin_adjusted_for_migration', чтобы различие между этим полем и страной на момент вручения объяснялось миграцией.
- Выявите страны с наибольшей долей "оттекших" и "притекших" лауреатов.
- Есть ли оттоки среди организаций?

**2.5. Многократные лауреаты**
- Найдите лауреатов, получившие больше одного приза. Кто получил больше 2 раз? 
- Совпадает ли категория второго приза с первой?  
- Сколько в среднем времени прошло между получением первой и второй премии?

**2.6 Отказы от премий**
- Определите, в каких категориях чаще отказывались от премий?
- Есть ли отказы среди организаций?

In [18]:
def origin_country(row):
    if row['type_'] == 'person':
        return row['country_birth']
    return row['country_founded']
def get_decade(row):
    return row['year']//10*10
def top_countries(rows):
    known = filter_rows(rows, lambda row: bool(row['country']))
    groups = group_by_attributes(known, ['country'])
    counts = []
    for key in sorted(groups):
        country = key[0]
        count = count_unique(groups[key], 'id')
        counts.append({'country': country, 'count': count})
    return top_n(counts, 'count', 5)
add_attribute(laureates_data, 'country', origin_country)
awards = []
for row in laureates_data:
    row_prizes = row['prizes_relevant']
    if row_prizes is None:
        continue
    for prize in row_prizes:
        year = prize['award_year']
        category = prize['category_en']
        if year is not None and category:
            awards.append({
                'id': row['id'],
                'country': row['country'],
                'category': category,
                'year': year,
            })

add_attribute(awards, 'decade', get_decade)

group_values = [
    ('По категориям', 'category'),
    ('По десятилетиям', 'decade'),
]

print('За весь период:', top_countries(laureates_data))

for title, value in group_values:
    print(title)
    groups = group_by_attributes(awards, [value])

    for key in sorted(groups):
        print(key[0], top_countries(groups[key]))

За весь период: [{'country': 'USA', 'count': 306}, {'country': 'United Kingdom', 'count': 96}, {'country': 'Germany', 'count': 82}, {'country': 'France', 'count': 63}, {'country': 'Japan', 'count': 32}]
По категориям
Chemistry [{'country': 'USA', 'count': 59}, {'country': 'United Kingdom', 'count': 26}, {'country': 'Germany', 'count': 24}, {'country': 'France', 'count': 12}, {'country': 'Japan', 'count': 9}]
Economic Sciences [{'country': 'USA', 'count': 56}, {'country': 'United Kingdom', 'count': 7}, {'country': 'Canada', 'count': 5}, {'country': 'France', 'count': 5}, {'country': 'the Netherlands', 'count': 4}]
Literature [{'country': 'France', 'count': 12}, {'country': 'USA', 'count': 10}, {'country': 'Sweden', 'count': 7}, {'country': 'United Kingdom', 'count': 6}, {'country': 'Italy', 'count': 5}]
Peace [{'country': 'USA', 'count': 26}, {'country': 'France', 'count': 11}, {'country': 'Switzerland', 'count': 8}, {'country': 'United Kingdom', 'count': 7}, {'country': 'Germany', 'cou

Мы видим, что  лидирует США с 306 лауреатамиЮ на втором места Uk, потом идет Германия. По В литературе лидирует Франциет 12 лауретоав против 10 у США. По десятелетиям лидер менялся: в 1900хд и 1910-х первой была Франция. 

In [17]:
first_awards = []
for person in people:
    birth_year = person['birth_year']
    row_prizes = person['prizes_relevant']
    if birth_year is None or row_prizes is None:
        continue
    first_year = None
    for prize in row_prizes:
        year = prize['award_year']
        if prize['prize_status'] == 'received' and year is not None:
            if first_year is None or year < first_year:
                first_year = year
                first_category = prize['category_en']
    if first_year is not None:
        first_awards.append({
            'age': first_year - birth_year,
            'year': first_year,
            'category': first_category or 'Не указана',
        })
add_attribute(first_awards, 'decade', get_decade)
def show_age(title, rows):
    stats = describe_values(rows, 'age')
    stats['mean'] = round(stats['mean'], 1)
    print(title, stats)
show_age('За весь период', first_awards)
for title, value in group_values:
    print(title)
    groups = group_by_attributes(first_awards, [value])
    for key in sorted(groups):
        show_age(key[0], groups[key])

За весь период {'count': 997, 'mean': 60.6, 'median': 61, 'max': 97}
По категориям
Chemistry {'count': 199, 'mean': 59.4, 'median': 58, 'max': 97}
Economic Sciences {'count': 101, 'mean': 67.2, 'median': 67, 'max': 90}
Literature {'count': 121, 'mean': 65.1, 'median': 67, 'max': 88}
Peace {'count': 111, 'mean': 61.0, 'median': 62, 'max': 87}
Physics {'count': 230, 'mean': 57.8, 'median': 56.0, 'max': 96}
Physiology or Medicine {'count': 235, 'mean': 59.0, 'median': 58, 'max': 87}
По десятилетиям
1900 {'count': 56, 'mean': 57.8, 'median': 58.0, 'max': 85}
1910 {'count': 37, 'mean': 53.1, 'median': 50, 'max': 80}
1920 {'count': 54, 'mean': 55.0, 'median': 56.0, 'max': 86}
1930 {'count': 55, 'mean': 51.6, 'median': 49, 'max': 75}
1940 {'count': 40, 'mean': 58.0, 'median': 56.0, 'max': 81}
1950 {'count': 70, 'mean': 54.1, 'median': 53.5, 'max': 79}
1960 {'count': 73, 'mean': 56.4, 'median': 57, 'max': 87}
1970 {'count': 101, 'mean': 59.3, 'median': 59, 'max': 87}
1980 {'count': 93, 'mean':

Среди 997 лауретов средний 997 лауретатов средний возраст составил 60.6, а медиана 61 год. Самый высокий средний рост в экономике 67 года, а самый низкий в физике 57.8 года. С 1950-х по 2010-у средний возраст рос в каждом следующем десятилетии, с 54 до 67

---------

## Бонусное задание на +2 балла


### Лингвистический анализ мотиваций**

**5.1. Анализ текстов мотиваций**
- Соберите корпус текстов мотиваций вручения премий. Для этого измените конфиг обработки призов.
- Выделите самые частые топ-50 слов. Какие из этих слов самые частые в принципе в английском языке (такие, как for, the и другие), а какие кажутся важными именно в контексте вручения Нобелевской премии (например, 'chemistry')?
- Самые частые слова, которые также являются частыми во всем английском языке занесите в отдельный конфиг-файл под названием `stopwords.py`. Вычистите их из текстов. Какие слова являются самыми частыми теперь? Сравните множества топ-10 самых частых слов по разным категориям


- Если есть желание, посчитайте TF-IDF + сравнените топ слов по этой метрике с просто самыми популярными словами)